# Quantum Autoencoder Anomaly Detector — Quickstart

Created by School of AI and School of QC.

This notebook runs a compact, leakage-safe experiment locally. It uses exact statevector simulation and does not claim quantum advantage.

In [ ]:
from quantum_autoencoder_anomaly_detector import ExperimentConfig, run_experiment
from quantum_autoencoder_anomaly_detector.inference import predict_frame

In [ ]:
config = ExperimentConfig(
    sample_count=400,
    anomaly_rate=0.10,
    normal_train_limit=40,
    ansatz_reps=1,
    optimizer_maxiter=14,
    output_root="artifacts/notebook",
).validate()
result = run_experiment(config, progress_callback=print)

In [ ]:
result.metrics[["model", "roc_auc", "average_precision", "recall", "business_cost"]]

## Inspect the learned compression

The model projects the encoded state onto the all-zero trash subspace, keeps the normalized latent state, restores zero trash qubits, and applies the inverse encoder. The round-trip state fidelity equals one minus the raw trash-probability anomaly score.

In [ ]:
states = result.prepared.amplitude_test[:5]
latent_states, compression_success = result.quantum_model.compress(states)
fidelity = result.quantum_model.reconstruction_fidelity(states)
latent_states.shape, compression_success, fidelity

In [ ]:
samples = result.partitions.full_dataset.iloc[:5]
predict_frame(result.output_directory, samples)